# Lab 2 | EDA — Khám phá dữ liệu

**Mục bình chọn:** EDA + tích hợp notebook (phần ngoài 5 lựa chọn bình chọn)

**Hạn bàn giao:** 24/10/2026, 20:00 cho phần EDA. Tích hợp ngày 27/10; rà soát cuối và báo cáo ngày 28/10.

> Bạn giúp người đọc hiểu hai bộ dữ liệu đang có gì: cột nào là thời gian, cột nào cần dự báo, dữ liệu có vấn đề gì và đồ thị trông như thế nào. Chưa cần fit ARIMA/SARIMA hay kết luận thay kiểm định ADF.

**Đối chiếu đề:** Bài cơ bản: Bài 1 trên Dataset 1. Nâng cao: phần vẽ chuỗi và nhận diện mùa vụ của Dataset 2. Nếu bạn đồng thời tích hợp, phần ghép bài là việc riêng sau khi nhận đủ các notebook.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần có:** hai CSV gốc, mô tả nguồn trong notebook tổng hợp và quy ước train/test.
**Không cần chờ ADF:** có thể bắt đầu ngay và làm song song với phần kiểm định.

## Cụ thể cần làm gì?

1. Đọc cả hai CSV, chuyển cột thời gian về kiểu ngày và đặt index thời gian theo tháng.
2. Kiểm tra thứ tự thời gian, trùng tháng, thiếu tháng, missing values và kiểu dữ liệu.
3. Ghi rõ tên dataset, cột mục tiêu, đơn vị, thời gian bắt đầu/kết thúc và số quan sát.
   Dataset 1 theo URL là a10; không gọi nhầm thành doanh số bán lẻ US.
4. Vẽ chuỗi theo thời gian; dùng phần train cho các nhận xét định hướng phân tích.
   Hình xem trước trong notebook tổng hợp chỉ là điểm bắt đầu, chưa phải nhận xét EDA của bạn.
5. Viết nhận xét riêng cho từng dataset về xu hướng, dấu hiệu mùa vụ và khả năng dừng sơ bộ.
   Chỉ ra bằng chứng nhìn thấy trên đồ thị, không khẳng định kết quả ADF khi chưa kiểm định.

## Cần bàn giao những gì?

- Notebook EDA chạy được, bảng kiểm tra chất lượng dữ liệu và đồ thị có nhãn/đơn vị rõ.
- Nhận xét Dataset 1 trả lời đủ: có xu hướng không, có mùa vụ không, có khả năng dừng không.
- Dataset 2 có nhận xét về chu kỳ theo tháng để hỗ trợ phần mùa vụ.
- Nếu phát hiện dữ liệu bất thường, báo cho các phần ADF/ACF/mô hình trước khi họ chốt kết quả.
- Khi tích hợp: ghép các cell theo thứ tự bài, thống nhất tên biến rồi Restart Kernel → Run All.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [ ] Hai dataset được mô tả đúng và kiểm tra chất lượng.
- [ ] Có đồ thị và nhận xét bằng lời, không chỉ dán ảnh.
- [ ] Giữ thống nhất dữ liệu/split cho những phần sau.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. Đọc dữ liệu, kiểm tra và trực quan hóa

Vẽ và mô tả xu hướng/mùa vụ trên train1.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1] if '__file__' in locals() else Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.lab2_data import load_datasets, chronological_split, SEASON_LENGTH

# 1. Đọc dữ liệu gốc và chia tập train/test theo đúng chuẩn 80/20 theo thời gian
raw1, raw2, y1, y2 = load_datasets()
train1, test1 = chronological_split(y1)
train2, test2 = chronological_split(y2)

# 2. Kiểm tra chất lượng dữ liệu
def check_quality(raw_df, series, name, target_col, unit):
    return {
        'Dataset': name,
        'Cột mục tiêu': target_col,
        'Đơn vị': unit,
        'Tổng N': len(series),
        'Bắt đầu': f'{series.index[0]:%Y-%m}',
        'Kết thúc': f'{series.index[-1]:%Y-%m}',
        'N (Train)': len(series) - int(len(series) * 0.2),
        'N (Test)': int(len(series) * 0.2),
        'Thiếu (NaN)': int(series.isna().sum()),
        'Trùng lặp ngày': bool(raw_df.iloc[:, 0].duplicated().any()),
        'Chu kỳ liên tục (MS)': series.index.freqstr == 'MS',
        'Tăng đơn điệu': bool(series.index.is_monotonic_increasing),
    }

quality_df = pd.DataFrame([
    check_quality(raw1, y1, 'Dataset 1 (a10)', 'value', 'Triệu AUD'),
    check_quality(raw2, y2, 'Dataset 2 (AirPassengers)', 'Passengers', 'Nghìn khách'),
])
display(quality_df)

# 3. Trực quan hóa EDA trên tập Train
fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# Dataset 1: Chuỗi train + Rolling mean (12 tháng)
axes[0, 0].plot(train1.index, train1.values, label='Train gốc (a10)', color='#1f77b4', lw=1.5)
axes[0, 0].plot(train1.index, train1.rolling(12).mean(), label='Rolling Mean (12 tháng)', color='#d62728', lw=2)
axes[0, 0].set_title('Dataset 1 (a10) — Xu hướng và chuỗi Train gốc')
axes[0, 0].set_ylabel('Triệu AUD')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Dataset 2: Chuỗi train + Rolling mean (12 tháng)
axes[0, 1].plot(train2.index, train2.values, label='Train gốc (Passengers)', color='#2ca02c', lw=1.5)
axes[0, 1].plot(train2.index, train2.rolling(12).mean(), label='Rolling Mean (12 tháng)', color='#d62728', lw=2)
axes[0, 1].set_title('Dataset 2 (AirPassengers) — Xu hướng và chuỗi Train gốc')
axes[0, 1].set_ylabel('Nghìn khách')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Dataset 1: Độ lệch chuẩn trượt (Rolling Std 12 tháng)
axes[1, 0].plot(train1.index, train1.rolling(12).std(), color='#ff7f0e', lw=1.8, label='Rolling Std (12 tháng)')
axes[1, 0].set_title('Dataset 1 (a10) — Độ lệch chuẩn trượt (Biên độ dao động tăng dần)')
axes[1, 0].set_ylabel('Std (Triệu AUD)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Dataset 2: Độ lệch chuẩn trượt (Rolling Std 12 tháng)
axes[1, 1].plot(train2.index, train2.rolling(12).std(), color='#9467bd', lw=1.8, label='Rolling Std (12 tháng)')
axes[1, 1].set_title('Dataset 2 (AirPassengers) — Độ lệch chuẩn trượt (Phương sai thay đổi)')
axes[1, 1].set_ylabel('Std (Nghìn khách)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Dataset 1: Phân bố mùa vụ theo tháng qua từng năm (Seasonal Month Plot)
for year, group in train1.groupby(train1.index.year):
    axes[2, 0].plot(group.index.month, group.values, marker='o', ms=3, alpha=0.5, label=str(year) if year in [1992, 1998, 2004] else None)
axes[2, 0].set_title('Dataset 1 (a10) — Mùa vụ: Đỉnh nhọn tháng 1, đáy sâu tháng 2')
axes[2, 0].set_xlabel('Tháng trong năm')
axes[2, 0].set_ylabel('Triệu AUD')
axes[2, 0].set_xticks(range(1, 13))
axes[2, 0].legend(title='Năm tiêu biểu')
axes[2, 0].grid(True, alpha=0.3)

# Dataset 2: Phân bố mùa vụ theo tháng qua từng năm
for year, group in train2.groupby(train2.index.year):
    axes[2, 1].plot(group.index.month, group.values, marker='s', ms=3, alpha=0.5, label=str(year) if year in [1949, 1953, 1957] else None)
axes[2, 1].set_title('Dataset 2 (AirPassengers) — Mùa vụ: Đỉnh du lịch hè (T7–T8), đáy T11')
axes[2, 1].set_xlabel('Tháng trong năm')
axes[2, 1].set_ylabel('Nghìn khách')
axes[2, 1].set_xticks(range(1, 13))
axes[2, 1].legend(title='Năm tiêu biểu')
axes[2, 1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


**Nhận xét (Kiểm tra chất lượng và trực quan hóa EDA):**

1. **Chất lượng dữ liệu:**
   - **Dataset 1 (a10):** Gồm 204 tháng (07/1991 – 06/2008). Tập train có 163 tháng (07/1991 – 01/2005), test có 41 tháng (02/2005 – 06/2008). Không có giá trị thiếu (NaN), không trùng lặp timestamp, chu kỳ tháng liên tục chuẩn `MS` và tăng đơn điệu theo thời gian.
   - **Dataset 2 (AirPassengers):** Gồm 144 tháng (01/1949 – 12/1960). Tập train có 115 tháng (01/1949 – 07/1958), test có 29 tháng (08/1958 – 12/1960). Dữ liệu hoàn toàn sạch, liên tục và đầy đủ.

2. **Đặc trưng trực quan trên tập Train:**
   - **Xu hướng (Trend):** Cả hai chuỗi đều thể hiện xu hướng tăng trưởng dài hạn rất rõ rệt (Upward trend). Đường Rolling Mean (12 tháng) dốc lên liên tục, giá trị trung bình không dừng qua các năm (D1 tăng từ ~3.5 triệu AUD lên hơn 20 triệu AUD; D2 tăng từ ~112 nghìn lên trên 400 nghìn hành khách).
   - **Mùa vụ (Seasonality):**
     + **D1 (a10):** Xuất hiện mô hình mùa vụ lặp lại chu kỳ 12 tháng với đặc thù *cực đại nhọn (spike)* vào **tháng 1** hằng năm (trung bình 11.41 triệu AUD) do cơ chế tích trữ thuốc trợ cấp đầu năm tài chính tại Úc, sau đó sụt giảm mạnh về mức đáy vào **tháng 2** (trung bình 6.41 triệu AUD).
     + **D2 (Passengers):** Chu kỳ mùa vụ 12 tháng cực kỳ chuẩn mực phản ánh ngành hàng không: Đạt đỉnh cao nhất vào mùa du lịch hè (**tháng 7 và tháng 8**, trung bình ~300 nghìn khách) và chạm đáy vào mùa đông (**tháng 11**, trung bình 192 nghìn khách).
   - **Phương sai (Variance):** Đồ thị Rolling Std cho thấy biên độ dao động mùa vụ tăng dần theo thời gian khi mức trung bình của chuỗi tăng lên $\rightarrow$ Dấu hiệu rõ ràng của chuỗi thời gian dạng nhân tính (Multiplicative time-series).

### 2. Nhận xét EDA và bàn giao

Trình bày nhận định sơ bộ về tính dừng; đối chiếu Dataset 2 nếu cần.

In [ ]:
# Tóm tắt các đặc trưng EDA và định hướng xử lý mô hình
eda_summary = pd.DataFrame([
    {
        'Dataset': 'D1 a10',
        'Xu hướng (Trend)': 'Tăng trưởng dài hạn mạnh mẽ (Upward trend)',
        'Mùa vụ (Seasonality)': 'Chu kỳ 12 tháng: Đỉnh nhọn T1 (11.41M), đáy sâu T2 (6.41M)',
        'Phương sai (Variance)': 'Tăng dần theo thời gian (Multiplicative)',
        'Đánh giá tính dừng': 'Không dừng (Non-stationary)',
        'Khuyến nghị mô hình': 'Cần sai phân d>=1 và sai phân mùa vụ D>=1 (m=12)',
    },
    {
        'Dataset': 'D2 Passengers',
        'Xu hướng (Trend)': 'Tăng trưởng tuyến tính ổn định',
        'Mùa vụ (Seasonality)': 'Chu kỳ 12 tháng: Đỉnh hè T7 (304.6K), đáy đông T11 (192.4K)',
        'Phương sai (Variance)': 'Biên độ dao động tăng tỷ lệ thuận với xu hướng',
        'Đánh giá tính dừng': 'Không dừng (Non-stationary)',
        'Khuyến nghị mô hình': 'Cần SARIMA với d=1, D=1, chu kỳ mùa vụ m=12',
    },
])
display(eda_summary)


**Nhận xét (Tổng kết EDA và Bàn giao cho các bài tiếp theo):**

**Trả lời các câu hỏi theo yêu cầu đề bài (Bài 1):**
- **Chuỗi có xu hướng không?**
  **Có.** Cả Dataset 1 và Dataset 2 đều có xu hướng tăng trưởng dài hạn rất mạnh mẽ (Upward trend). Trung bình chuỗi thay đổi theo thời gian, thể hiện qua đường trung bình trượt 12 tháng (Rolling Mean) tăng liên tục qua các năm.
- **Có dấu hiệu mùa vụ không?**
  **Có.** Cả hai dataset đều có mùa vụ chu kỳ $m = 12$ tháng lặp đi lặp lại hàng năm:
  + *Dataset 1:* Đỉnh nhọn bất thường vào tháng 1 và đáy rơi vào tháng 2.
  + *Dataset 2:* Đỉnh du lịch cao điểm hè vào tháng 7–8 và đáy vào tháng 11.
- **Chuỗi có khả năng dừng không?**
  **Không dừng (Non-stationary).** Dấu hiệu vi phạm tính dừng bao gồm: (1) Giá trị kỳ vọng $\mu_t$ không cố định mà tăng theo xu hướng; (2) Phương sai $\sigma^2_t$ tăng dần theo thời gian (biên độ dao động rộng dần); (3) Tồn tại tương quan mùa vụ chu kỳ 12 tháng.

**Bàn giao định hướng cho các bài tiếp theo:**
1. **Bàn giao sang Bài 2–3 (ADF & Sai phân):**
   - Dự báo kiểm định ADF trên chuỗi gốc sẽ không bác bỏ $H_0$ ($p$-value sẽ cao, kết luận chuỗi không dừng).
   - Cần thực hiện sai phân thường ít nhất một lần ($d = 1$) để triệt tiêu xu hướng, và bắt buộc phải thực hiện thêm sai phân mùa vụ bậc 1 ($D = 1, m = 12$) để loại bỏ hoàn toàn tính chu kỳ mùa vụ hàng năm.
2. **Bàn giao sang Bài 4 & 5 (ACF/PACF, ARIMA, SARIMA):**
   - Vì cả hai chuỗi đều có thành phần mùa vụ rất mạnh, mô hình ARIMA thuần túy (không có mùa vụ) sẽ gặp hạn chế lớn khi nắm bắt các đỉnh/đáy lặp lại hàng năm.
   - Dataset 2 được khuyến nghị ứng dụng mô hình **SARIMA$(p, d, q)(P, D, Q)_{12}$** để dự báo chính xác tính chất mùa vụ mùa hè/mùa đông.